<a href="https://colab.research.google.com/github/suryasai99/Object_detection/blob/main/Detector_architectures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torchinfo

In [2]:
import os, requests, cv2, torch
from zipfile import ZipFile
import torchvision.transforms as T
from torchvision.models import resnet50
from torchinfo import summary

## Download dependencies

In [3]:
def download_file(url, save_name):
    """
    "Download and save the file."

    arguments:
    url (str): URL path of the file.
    save_name: (str): file path to save the downloaded file.
    """
    file = requests.get(url)
    open(save_name, 'wb').write(file.content)
    print(f"Downloaded {save_name}...")
    return

In [4]:
def unzip(zip_file_path=None):
    """
    "Unzip the file"

    arguments:
    zip_file_path (str): The zipped file path

    """
    try:
        with ZipFile(zip_file_path) as z:
            z.extractall("./")
            print(f"Extracted {zip_file_path}...\n")
    except:
        print("Invalid file")

    return

In [5]:
if not os.path.exists('models'):
    download_file(
                  'https://www.dropbox.com/s/0x6xo687g3or23g/detector_nn_architecture.zip?dl=1',
                  'detector_nn_architecture.zip'
                 )

    unzip('detector_nn_architecture.zip')

Downloaded detector_nn_architecture.zip...
Extracted detector_nn_architecture.zip...



In [8]:
os.listdir('./')

['.config',
 'detector.py',
 'test_images',
 'detector_nn_architecture.zip',
 'drive',
 'fpn.py',
 'sample_data']

# Resnet50 backbone

In [9]:
resnet50_feature_extractor = resnet50(weights = "DEFAULT")

print(summary(resnet50_feature_extractor, input_size = (1, 3, 300, 300)))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 113MB/s]


Layer (type:depth-idx)                   Output Shape              Param #
ResNet                                   [1, 1000]                 --
├─Conv2d: 1-1                            [1, 64, 150, 150]         9,408
├─BatchNorm2d: 1-2                       [1, 64, 150, 150]         128
├─ReLU: 1-3                              [1, 64, 150, 150]         --
├─MaxPool2d: 1-4                         [1, 64, 75, 75]           --
├─Sequential: 1-5                        [1, 256, 75, 75]          --
│    └─Bottleneck: 2-1                   [1, 256, 75, 75]          --
│    │    └─Conv2d: 3-1                  [1, 64, 75, 75]           4,096
│    │    └─BatchNorm2d: 3-2             [1, 64, 75, 75]           128
│    │    └─ReLU: 3-3                    [1, 64, 75, 75]           --
│    │    └─Conv2d: 3-4                  [1, 64, 75, 75]           36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 75, 75]           128
│    │    └─ReLU: 3-6                    [1, 64, 75, 75]           --
│ 

In [10]:
inputs = torch.rand((1, 3, 300, 300))

In [11]:
x = resnet50_feature_extractor.conv1(inputs)
x = resnet50_feature_extractor.bn1(x)
x = resnet50_feature_extractor.relu(x)
x = resnet50_feature_extractor.maxpool(x)

layer1_output = resnet50_feature_extractor.layer1(x)
layer2_output = resnet50_feature_extractor.layer2(layer1_output)
layer3_output = resnet50_feature_extractor.layer3(layer2_output)
layer4_output = resnet50_feature_extractor.layer4(layer3_output)

print(layer2_output.shape)
print(layer3_output.shape)
print(layer4_output.shape)


torch.Size([1, 512, 38, 38])
torch.Size([1, 1024, 19, 19])
torch.Size([1, 2048, 10, 10])


# The detector architecture

In [12]:
from detector import Detector

In [13]:
backbone_name = "resnet50"
num_classes = 2
fpn_channels = 64
num_anchors = 9

In [14]:
detector = Detector(
    backbone_name,
    num_classes,
    fpn_channels,
    num_anchors
)

In [15]:
loc_preds, cls_heads = detector(inputs)

In [16]:
print(f"localization head shape: {loc_preds.shape}")
print(f"classification head shape: {cls_heads.shape}")

localization head shape: torch.Size([1, 17451, 4])
classification head shape: torch.Size([1, 17451, 2])


## A sample inference

In [17]:
img1 = cv2.imread("./test_images/FudanPed00001.png")[...,::-1]
img2 = cv2.imread("./test_images/FudanPed00002.png")[...,::-1]

img1_resize = cv2.resize(img1, (300,300))
img2_resize = cv2.resize(img2, (300,300))

mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

mean = torch.Tensor(mean)
std = torch.Tensor(std)

transforms = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

img1_t = transforms(img1_resize)
img2_t = transforms(img2_resize)

inputs = torch.stack([img1_t, img2_t])

In [18]:
# Already initialized the detector model. Now let's predict
preds = detector(inputs)

print(f"Location Preds size: {preds[0].shape}")
print(f"Class Preds size: {preds[1].shape}")

Location Preds size: torch.Size([2, 17451, 4])
Class Preds size: torch.Size([2, 17451, 2])
